new


In [ ]:
# === 1. Install correct dependencies ===
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.2" "bitsandbytes>=0.43.1" peft sentencepiece

# === 2. Import libraries ===
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# === 3. Model info ===
model_id = "tanvir211/falcon-7b-lora-merged"

# === 4. Quantization config (for 4-bit model) ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# === 5. Load model and tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

# === 6. Inference ===
prompt = "what are the main types of research?."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
def generate_text(prompt, max_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [ ]:
!ngrok authtoken 349OJ38Y1W1JWZMMIHvmJ0Rjxcj_2broU4qKJFQegSowyQ27y

In [ ]:
# Inference function
def generate_text(prompt, max_tokens=256, temperature=0.7, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
from fastapi.middleware.cors import CORSMiddleware





In [ ]:
!pip install langchain sentence-transformers faiss-cpu PyPDF2

In [ ]:
import re

def clean_context(text):
    # Remove page numbers like "13 | Page" or "18 | Page"
    text = re.sub(r'\d+\s*\|\s*Page', '', text)
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [ ]:
def extract_answer(llm_output: str) -> str:
    """
    Given LLM output like:
    'Answer: - The title is "Ethical Leadership Traits and Legacy of Satya Nadella".'
    returns only the text after 'Answer:' and strips extra spaces and quotes.
    """
    # Split on 'Answer:'
    parts = llm_output.split("Answer:")
    if len(parts) > 1:
        answer = parts[1].strip()
        # Optionally remove leading hyphens or quotes
        answer = re.sub(r'^[-"\']+\s*', '', answer)
        answer = re.sub(r'["\']+$', '', answer)
        return answer
    else:
        # fallback if "Answer:" is missing
        return llm_output.strip()


In [ ]:
!pip install pymupdf sentence-transformers faiss-cpu


# Main Backend

In [ ]:
import os
import re
import tempfile
import numpy as np
import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import fitz  # PyMuPDF

In [ ]:
# ==============================================================
# FastAPI setup
# ==============================================================
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
# ==============================================================
# Globals
# ==============================================================
index = None
doc_mapping = {}

In [ ]:
# ==============================================================
# Load embedding model and LLM
# ==============================================================
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

model_id = "tanvir211/falcon-7b-lora-merged"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

In [ ]:
# ==============================================================
# Helper functions
# ==============================================================
def clean_context(text):
    text = re.sub(r'\d+\s*\|\s*Page', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_answer(llm_output: str) -> str:
    parts = llm_output.split("Answer:")
    if len(parts) > 1:
        answer = parts[1].strip()
        answer = re.sub(r'^[-"\']+\s*', '', answer)
        answer = re.sub(r'["\']+$', '', answer)
        return answer
    else:
        return llm_output.strip()

def generate_rag(query: str, detail_level: str = "concise", max_tokens: int = 300, top_k: int = 3) -> str:
    global index, doc_mapping, embedding_model, tokenizer, model

    if index is None or len(doc_mapping) == 0:
        return "No documents indexed yet. Please check your Drive folder."

    q_vec = np.array([embedding_model.encode([query])[0]]).astype("float32")
    D, I = index.search(q_vec, top_k)

    valid_indices = [i for i in I[0] if i != -1]
    if not valid_indices:
        return "No relevant documents found."

    retrieved_docs = [doc_mapping[i] for i in valid_indices]
    context_cleaned = clean_context(" ".join(retrieved_docs))

    prompt = f"""
You are a highly intelligent assistant specialized in extracting and explaining information from documents.
Use ONLY the context provided below to answer the user’s question.
Do not hallucinate or add information not present in the context.

Instructions:
1. If the user asks for a concise answer, give a short, precise response.
2. If the user asks for a detailed explanation, provide multiple sentences or structured details.
3. Always answer clearly and in full sentences.
4. Remove any page numbers or metadata.
5. Do NOT repeat the question.

Context:
{context_cleaned}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=(detail_level=="detailed"),
        temperature=0.7 if detail_level=="detailed" else 0.0,
        top_p=0.9
    )
    llm_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_answer(llm_output)

# ==============================================================
# NEW: Load PDFs from Google Drive
# ==============================================================
def load_pdfs_from_drive(base_folder="/content/drive/MyDrive/CDU"):
    global index, doc_mapping
    pdf_paths = []

    # --- Collect all PDF files (recursive) ---
    for root, _, files in os.walk(base_folder):
        for file in files:
            if file.lower().endswith(".pdf"):
                pdf_paths.append(os.path.join(root, file))

    if not pdf_paths:
        print("⚠️ No PDFs found in Drive folder.")
        return

    doc_texts = []
    for path in pdf_paths:
        try:
            pdf_doc = fitz.open(path)
            for page in pdf_doc:
                text = page.get_text()
                if text.strip():
                    doc_texts.append(text)
            pdf_doc.close()
        except Exception as e:
            print(f"Error reading {path}: {e}")

    embeddings = embedding_model.encode(doc_texts, convert_to_numpy=True).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    doc_mapping = {i: doc_texts[i] for i in range(len(doc_texts))}

    print(f"✅ Indexed {len(doc_texts)} pages from {len(pdf_paths)} PDFs in Drive.")


In [ ]:
# ==============================================================
# FastAPI models and routes
# ==============================================================
class GenerateRequest(BaseModel):
    prompt: str
    detail_level: str = "concise"
    max_tokens: int = 300


@app.post("/generate")
def generate(request: GenerateRequest):
    try:
        response = generate_rag(request.prompt, request.detail_level, request.max_tokens)
        return {"response": response}
    except Exception as e:
        return {"error": str(e)}


In [ ]:
# ==============================================================
# Ngrok + Server
# ==============================================================
public_url = ngrok.connect(8000)
print("Public API URL:", public_url)

import nest_asyncio
import asyncio
from uvicorn import Config, Server

nest_asyncio.apply()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Load the PDFs automatically on startup ---
load_pdfs_from_drive("/content/drive/MyDrive/AI")

config = Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = Server(config)

asyncio.get_event_loop().run_until_complete(server.serve())


✅ Indexed 427 pages from 1 PDFs in Drive.


INFO:     Started server process [324]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down


RuntimeError: Event loop stopped before Future completed.

# Skip